In [ ]:
import pandas as pd
import tiktoken
from tqdm import tqdm








# To get the tokeniser corresponding to a specific model in the OpenAI API:
enc = tiktoken.get_encoding("o200k_base")

enc.encode("tiktoken is great!")

In [ ]:
def token_counter(text):
    return len(enc.encode(text))


print(token_counter("i love cows"))




In [ ]:
csv_paths = ["/Users/josh/Documents/symptom-extraction-demo/data/status_data.csv"]

In [ ]:
tqdm.pandas()
total_count=0
for _ in csv_paths:
    #df=pd.read_csv(_)
    df=pd.read_csv(_)
    df['tokens'] = df['text'].progress_apply(token_counter)
    count=df['tokens'].sum()
    total_count += count


print ("!!! done !!!!")

In [ ]:
print('millions of tokens')
print(total_count/ 1_000_000)
print('cost in $')
print((total_count/1_000_000)*2)

In [1]:
from openai import OpenAI
from pydantic import BaseModel
import os
from dotenv import load_dotenv
from src.schema import OutputSchema
from src.llm import openai_chat_completion_response


# create client
"""client = OpenAI(
    api_key='ollama',
    max_retries=3,
    # if using openai use base url 'https://api.openai.com/v1'
    base_url='http://localhost:11434/v1'
)"""

# Load .env into environment
load_dotenv()
# Pull API key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Please set the OPENAI_API_KEY environment variable in your .env file")


client = OpenAI(
    api_key=api_key,
    max_retries=3,
)

In [3]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5-nano-2025-08-07",
    input="Write a one paragraph about the paper called an abundence of possums"
)

print(response.output_text)

Here’s a sample one-paragraph summary you can use as a blurb or starting point:

An Abundance of Possums investigates the population density, distribution, and ecological role of possums across rural and urban landscapes. The study combines long-term field surveys, trap-and-release data, and camera-trap footage with population modeling to estimate abundance and track changes over time. It explores drivers such as food resources, habitat connectivity, predation by foxes and feral cats, disease, and climate variability, showing how different habitats support varying levels of possum presence. The analysis suggests that possums can thrive in mosaics of native and modified habitats, but fragmentation and management practices can shift dynamics in unexpected ways, with implications for biodiversity, forest regeneration, and urban ecology. The paper concludes with recommendations for monitoring, habitat management, and evidence-based policies to balance possum populations with ecosystem heal

In [ ]:
print(type(research_paper1))
print(type(research_paper))

In [ ]:
class ResearchPaperExtraction(BaseModel):
    title: str

  

response = client.responses.parse(
    model="gpt-4.1-nano-2025-04-14",
    input=[
        {
            "role": "system",
            "content": "You are an expert at structured data extraction. the paper is called an abundence of possums"},
        {"role": "user", "content": "..."},
    ],
    text_format=ResearchPaperExtraction,
)

research_paper = response.output_text
research_paper1 = json.loads(research_paper)

In [ ]:
print(type(research_paper1))
print(type(research_paper))

In [ ]:
print(research_paper1)

In [ ]:
from openai import OpenAI

client = OpenAI()

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia"
                }
            },
            "required": [
                "location"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano-2025-04-14",
    messages=[{"role": "user", "content": "What is the weather like in Paris today?"}],
    tools=tools
)

print(completion.choices[0].message.tool_calls)

In [ ]:
import json

output = completion.choices[0].message.tool_calls[0].function.arguments
output = json.loads(output)
print(output)

In [ ]:
from pydantic import BaseModel
from openai import OpenAI
import os, json

client = OpenAI(api_key=os.environ.get("TOGETHER_API_KEY"),
                base_url="https://api.together.xyz/v1",)



completion = client.chat.completions.create(
    model="Qwen/Qwen3-235B-A22B-fp8-tput",
    messages=[
        {"role": "system", "content": "Extract the name of the paper information think about it first."},
        {"role": "user", "content": "There is a paper called an abundence of possums"},
    ],
    response_format={
            "type": "json_schema",
            "schema": ResearchPaperExtraction.model_json_schema(),
        },
    )

output = json.loads(completion.choices[0].message.content)
pretty_output = json.dumps(output, indent=2)


In [ ]:
print(research_paper1)

In [ ]:
print(type(output))
print(type(pretty_output))

In [ ]:
print(completion)

In [ ]:
import dspy
lm = dspy.LM("openai/gpt-4.1-nano-2025-04-14", api_key=api_key)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal

class Classify(dspy.Signature):
    """symptom discussion -> presence/absence of symptom talk"""

    excerpt: str = dspy.InputField()
    symptom: Literal["Positive", "Negative"] = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(exerpt="This book was super fun to read, though not the last chapter.")

In [ ]:
import pandas as pd
import dspy
df = pd.read_csv('data/status_data.csv').head(200)
devset = [dspy.Example(excerpt=row['text'], symptom=row['label']).with_inputs('excerpt') for _, row in df.iterrows()]
def metric(example, pred, trace=None):
    return pred.symptom == example.symptom
from dspy.evaluate import Evaluate
evaluate = Evaluate(devset=devset, metric=metric)
evaluate(classify)

In [ ]:
# Import the optimizer
from dspy.teleprompt import MIPROv2

# Initialize optimizer
teleprompter = MIPROv2(
    metric=metric,
    auto="heavy", # Can choose between light, medium, and heavy optimization runs
)

# Optimize program
print(f"Optimizing program with MIPRO...")
optimized_program = teleprompter.compile(
    classify.deepcopy(),
    trainset=devset,
    max_bootstrapped_demos=2,
    max_labeled_demos=0,
    requires_permission_to_run=False,
)

# Save optimize program for future use
#optimized_program.save(f"mipro_optimized")

# Evaluate optimized program
print(f"Evaluate optimized program...")
evaluate(optimized_program, devset=devset[:])

In [ ]:
dspy.inspect_history(n=1)

In [5]:
# Combine DataFrames by aligning on row index
import pandas as pd

df1 = pd.read_csv('responses/status_responses_4_1_mini_4_1.csv')
df2 = pd.read_csv('responses/status_responses_Llama_3_1_8B_Instruct_Turbo.csv')
df3 = pd.read_csv('responses/status_responses_Llama_3_3_70B_Instruct_Turbo.csv')
df4 = pd.read_csv('responses/status_responses_kimi_deepseek.csv')
df5 = pd.read_csv('responses/status_responses_41nano_40mini.csv')

dfs = [df1, df2, df3, df4, df5]

# Concatenate horizontally and remove duplicate columns
merged_df = pd.concat(dfs, axis=1)

merged_df.head()

,text,source,label,pred_status_gpt_4_1_mini_2025_04_14,pred_status_gpt_4_1_2025_04_14,text,source,label,pred_status_meta_llama/Meta_Llama_3_1_8B_Instruct_Turbo,text,...,text,source,label,pred_status_deepseek_ai/DeepSeek_V3,pred_status_moonshotai/Kimi_K2_Instruct,text,source,label,pred_status_gpt_4_1_nano_2025_04_14,pred_status_gpt_4o_mini_2024_07_18
0,"D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,Positive,True,True,"D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,Positive,True,"D: How may I help you?\n\nP: Hi, umm, so I've ...",...,"D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,Positive,True,True,"D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,Positive,True,True
1,D: OK. And how about your appetite? Has that c...,RES0181.txt,Positive,True,True,D: OK. And how about your appetite? Has that c...,RES0181.txt,Positive,True,D: OK. And how about your appetite? Has that c...,...,D: OK. And how about your appetite? Has that c...,RES0181.txt,Positive,True,True,D: OK. And how about your appetite? Has that c...,RES0181.txt,Positive,True,True
2,D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,Positive,False,False,D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,Positive,True,D: OK. Have you had any any headaches?\n\nP: U...,...,D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,Positive,True,False,D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,Positive,False,False
3,D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,Positive,True,True,D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,Positive,True,D: Have you had any abdominal pain?\n\nP: No. ...,...,D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,Positive,True,True,D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,Positive,True,True
4,"P: I use an insulin pump, yeah.\n\nD: OK, exce...",RES0181.txt,Positive,True,True,"P: I use an insulin pump, yeah.\n\nD: OK, exce...",RES0181.txt,Positive,True,"P: I use an insulin pump, yeah.\n\nD: OK, exce...",...,"P: I use an insulin pump, yeah.\n\nD: OK, exce...",RES0181.txt,Positive,True,True,"P: I use an insulin pump, yeah.\n\nD: OK, exce...",RES0181.txt,Positive,True,True


In [13]:


merged_df.columns



Index(['text', 'source', 'label', 'pred_status_gpt_4_1_mini_2025_04_14',
       'pred_status_gpt_4_1_2025_04_14', 'text', 'source', 'label',
       'pred_status_meta_llama/Meta_Llama_3_1_8B_Instruct_Turbo', 'text',
       'source', 'label', 'pred_status_meta_llama/Llama_3_2_3B_Instruct_Turbo',
       'pred_status_meta_llama/Llama_3_3_70B_Instruct_Turbo', 'text', 'source',
       'label', 'pred_status_deepseek_ai/DeepSeek_V3',
       'pred_status_moonshotai/Kimi_K2_Instruct', 'text', 'source', 'label',
       'pred_status_gpt_4_1_nano_2025_04_14',
       'pred_status_gpt_4o_mini_2024_07_18'],
      dtype='object')

In [ ]:
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]



merged_df.shape

# save as csv
merged_df.to_csv('responses/status_responses_all.csv', index=False)


#do the same for detail_responses





In [18]:
df1 = pd.read_csv('responses/detail_responses_4_1_mini_4_1.csv')
df2 = pd.read_csv('responses/detail_responses_Llama_3_1_8B_Instruct_Turbo.csv')
df3 = pd.read_csv('responses/detail_responses_Llama_3_3_70B_Instruct_Turbo.csv')
df4 = pd.read_csv('responses/detail_responses_kimi_deepseek.csv')
df5 = pd.read_csv('responses/detail_responses_41nano_40mini.csv')

dfs = [df1, df2, df3, df4, df5]

# Concatenate horizontally and remove duplicate columns
merged_df = pd.concat(dfs, axis=1)

merged_df.head()

,text,source,label,response_detail_gpt_4_1_mini_2025_04_14,response_detail_gpt_4_1_2025_04_14,text,source,label,response_detail_meta_llama/Meta_Llama_3_1_8B_Instruct_Turbo,text,...,text,source,label,response_detail_deepseek_ai/DeepSeek_V3,response_detail_moonshotai/Kimi_K2_Instruct,text,source,label,response_detail_gpt_4_1_nano_2025_04_14,response_detail_gpt_4o_mini_2024_07_18
0,"D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,fever;other;trouble drinking fluids,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...","D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,fever;other;trouble drinking fluids,"{'Anxiety': False, 'Concentration_Problems': F...","D: How may I help you?\n\nP: Hi, umm, so I've ...",...,"D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,fever;other;trouble drinking fluids,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...","D: How may I help you?\n\nP: Hi, umm, so I've ...",RES0181.txt,fever;other;trouble drinking fluids,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F..."
1,D: OK. And how about your appetite? Has that c...,RES0181.txt,fever;other;poor appetite,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...",D: OK. And how about your appetite? Has that c...,RES0181.txt,fever;other;poor appetite,"{'Anxiety': False, 'Concentration_Problems': F...",D: OK. And how about your appetite? Has that c...,...,D: OK. And how about your appetite? Has that c...,RES0181.txt,fever;other;poor appetite,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...",D: OK. And how about your appetite? Has that c...,RES0181.txt,fever;other;poor appetite,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F..."
2,D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,cough;headache;nausea;other;pain;shortness of ...,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...",D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,cough;headache;nausea;other;pain;shortness of ...,"{'Anxiety': False, 'Concentration_Problems': F...",D: OK. Have you had any any headaches?\n\nP: U...,...,D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,cough;headache;nausea;other;pain;shortness of ...,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...",D: OK. Have you had any any headaches?\n\nP: U...,RES0181.txt,cough;headache;nausea;other;pain;shortness of ...,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F..."
3,D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,constipation;diarrhea;fatigue;other;pain;rash,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...",D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,constipation;diarrhea;fatigue;other;pain;rash,"{'Anxiety': False, 'Concentration_Problems': F...",D: Have you had any abdominal pain?\n\nP: No. ...,...,D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,constipation;diarrhea;fatigue;other;pain;rash,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F...",D: Have you had any abdominal pain?\n\nP: No. ...,RES0181.txt,constipation;diarrhea;fatigue;other;pain;rash,"{'Anxiety': False, 'Concentration_Problems': F...","{'Anxiety': False, 'Concentration_Problems': F..."
4,"P: I use an insulin pump, yeah.\n\nD: OK, exce...",RES0181.txt,anxiety,"{'Anxiety': True, 'Concentration_Problems': Fa...","{'Anxiety': True, 'Concentration_Problems': Fa...","P: I use an insulin pump, yeah.\n\nD: OK, exce...",RES0181.txt,anxiety,"{'Anxiety': True, 'Concentration_Problems': Fa...","P: 

In [21]:
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]



merged_df.shape

# save as csv
merged_df.to_csv('responses/detail_responses_all.csv', index=False)

In [22]:
merged_df.shape
merged_df.columns

Index(['text', 'source', 'label', 'response_detail_gpt_4_1_mini_2025_04_14',
       'response_detail_gpt_4_1_2025_04_14',
       'response_detail_meta_llama/Meta_Llama_3_1_8B_Instruct_Turbo',
       'response_detail_meta_llama/Llama_3_2_3B_Instruct_Turbo',
       'response_detail_meta_llama/Llama_3_3_70B_Instruct_Turbo',
       'response_detail_deepseek_ai/DeepSeek_V3',
       'response_detail_moonshotai/Kimi_K2_Instruct',
       'response_detail_gpt_4_1_nano_2025_04_14',
       'response_detail_gpt_4o_mini_2024_07_18'],
      dtype='object')

In [ ]:
"gpt_4_1_mini_2025_04_14","gpt_4_1_2025_04_14","meta_llama/Meta_Llama_3_1_8B_Instruct_Turbo", "meta_llama/Llama_3_2_3B_Instruct_Turbo","meta_llama/Llama_3_3_70B_Instruct_Turbo","deepseek_ai/DeepSeek_V3", "moonshotai/Kimi_K2_Instruct", "gpt_4_1_nano_2025_04_14", "gpt_4o_mini_2024_07_18"